# Advanced Multi-Task: MMoE & PLE on Ali-CCP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/rec_system_experimental/blob/main/notebooks/04_aliccp_cvr/03_advanced_cvr.ipynb)
[![MMoE Paper](https://img.shields.io/badge/Paper-KDD%202018-blue)](https://dl.acm.org/doi/10.1145/3219819.3220007)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand MMoE architecture: shared experts with per-task gating networks
2. Understand PLE architecture: task-specific + shared experts with progressive extraction
3. Implement both models from scratch in PyTorch
4. Train and evaluate on Ali-CCP, comparing against ESMM
5. Analyze expert gating patterns and task specialization
6. Perform ablation studies on number of experts and extraction layers

## Prerequisites

- Completed Notebooks 01-02
- Preprocessed data at `data/aliccp/processed/aliccp_processed.pkl`
- ESMM results at `data/aliccp/processed/esmm_results.json`

## Table of Contents

1. [Theory: Multi-Task Expert Networks](#1-theory-multi-task-expert-networks)
2. [Setup & Data Loading](#2-setup--data-loading)
3. [Shared Components](#3-shared-components)
4. [MMoE Implementation](#4-mmoe-implementation)
5. [PLE Implementation](#5-ple-implementation)
6. [Training](#6-training)
7. [Comparison with ESMM](#7-comparison-with-esmm)
8. [Expert Utilization Analysis](#8-expert-utilization-analysis)
9. [Ablation Studies](#9-ablation-studies)
10. [Exercises](#exercises)
11. [Summary & Key Takeaways](#summary--key-takeaways)

In [ ]:
import os
import json
import time
import pickle
import warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, log_loss, roc_curve

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Plotting
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Paths
DATA_DIR = Path('../../data/aliccp')
PROCESSED_DIR = DATA_DIR / 'processed'

# Hyperparameters
EMBEDDING_DIM = 8
N_EXPERTS = 4
EXPERT_DIM = 128
TOWER_DIMS = [64, 32]
BATCH_SIZE = 4096
LEARNING_RATE = 1e-3
NUM_EPOCHS = 5
DROPOUT = 0.2

## 1. Theory: Multi-Task Expert Networks

### 1.1 MMoE (Ma et al., Google, KDD 2018)

Multi-gate Mixture-of-Experts (MMoE) addresses a key limitation of shared-bottom models:
when tasks are loosely related, a shared representation can hurt performance.

**Architecture:**
- $N$ **expert networks** $\{E_1, E_2, ..., E_N\}$: each processes the full input
- **Task-specific gates** $\{G_1, G_2\}$: softmax over expert outputs
- **Task towers**: final predictions from gated expert mixtures

$$G_k(x) = \text{softmax}(W_{gate}^k \cdot x)$$
$$f_k(x) = \text{Tower}_k\left(\sum_{i=1}^{N} G_k^{(i)}(x) \cdot E_i(x)\right)$$

> **Concept:** Each task can dynamically weight which experts to rely on. If CTR and CVR
> need different feature representations, the gating networks learn to route them to
> different expert combinations.

### 1.2 PLE (Tang et al., Tencent, RecSys 2020)

Progressive Layered Extraction (PLE) improves on MMoE by:
1. Adding **task-specific experts** alongside shared experts
2. Using **progressive extraction layers** that refine representations

**Architecture per extraction layer:**
- $M$ **shared experts**: used by all tasks
- $M_k$ **task-specific experts**: used only by task $k$
- **Task gates**: combine shared + task-specific expert outputs

$$G_k^{(l)}(x) = \text{softmax}(W_k^{(l)} \cdot h_k^{(l-1)})$$

$$h_k^{(l)} = \sum_{i} G_k^{(l,i)} \cdot [E_{shared}^{(l,i)}(\cdot), E_{task_k}^{(l,i)}(\cdot)]$$

> **Pro Tip:** PLE reduces **negative transfer** -- the phenomenon where training on one
> task hurts another. Task-specific experts ensure each task has dedicated capacity,
> while shared experts still enable cross-task knowledge transfer.

## 2. Setup & Data Loading

In [ ]:
# Load preprocessed data
processed_path = PROCESSED_DIR / 'aliccp_processed.pkl'

if not processed_path.exists():
    raise FileNotFoundError(
        f"Processed data not found at {processed_path.resolve()}. "
        "Please run Notebook 01 first."
    )

with open(processed_path, 'rb') as f:
    data = pickle.load(f)

field_cardinalities = data['field_cardinalities']
all_fields = data['all_fields']
field_dims = [field_cardinalities[f] for f in all_fields]

print(f'Train: {data["X_ids_train"].shape[0]:,}, Val: {data["X_ids_val"].shape[0]:,}')
print(f'Fields: {len(all_fields)}, Dims: {field_dims}')
print(f'Train CTR: {data["y_click_train"].mean()*100:.2f}%, CTCVR: {data["y_conv_train"].mean()*100:.4f}%')

In [ ]:
# Dataset and DataLoader
X_ids_train = torch.LongTensor(data['X_ids_train'])
X_vals_train = torch.FloatTensor(data['X_vals_train'])
y_click_train = torch.FloatTensor(data['y_click_train'])
y_conv_train = torch.FloatTensor(data['y_conv_train'])
X_ids_val = torch.LongTensor(data['X_ids_val'])
X_vals_val = torch.FloatTensor(data['X_vals_val'])
y_click_val = torch.FloatTensor(data['y_click_val'])
y_conv_val = torch.FloatTensor(data['y_conv_val'])

train_ds = TensorDataset(X_ids_train, X_vals_train, y_click_train, y_conv_train)
val_ds = TensorDataset(X_ids_val, X_vals_val, y_click_val, y_conv_val)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 3. Shared Components

Both MMoE and PLE use the same embedding layer and ESMM-style CTCVR = CTR x CVR formulation.

In [ ]:
def embed_features(embeddings, feat_ids, feat_vals=None):
    """Shared embedding logic for all models.
    
    Args:
        embeddings: nn.ModuleList of embedding tables
        feat_ids: (B, n_fields) LongTensor
        feat_vals: (B, n_fields) FloatTensor, optional
    
    Returns:
        (B, n_fields * embed_dim) concatenated embeddings
    """
    embs = []
    for i, emb in enumerate(embeddings):
        e = emb(feat_ids[:, i])
        if feat_vals is not None:
            e = e * feat_vals[:, i:i+1]
        embs.append(e)
    return torch.cat(embs, dim=1)


def make_tower(input_dim, hidden_dims, dropout=0.2):
    """Build an MLP tower: [Linear -> ReLU -> Dropout] x N -> Linear."""
    layers = []
    prev = input_dim
    for dim in hidden_dims:
        layers.extend([nn.Linear(prev, dim), nn.ReLU(), nn.Dropout(dropout)])
        prev = dim
    layers.append(nn.Linear(prev, 1))
    return nn.Sequential(*layers)

## 4. MMoE Implementation

### 4.1 Expert and Gating Networks

In [ ]:
class MMoE(nn.Module):
    """Multi-gate Mixture-of-Experts for CVR prediction on Ali-CCP.
    
    Uses ESMM-style CTCVR = CTR x CVR to address sample selection bias.
    
    Architecture:
        Shared Embeddings -> [Expert1, ..., ExpertN] -> Gate_CTR, Gate_CVR
        -> Tower_CTR(gated_ctr) -> ctr_pred
        -> Tower_CVR(gated_cvr) -> cvr_pred
        -> ctcvr = ctr * cvr
    
    Args:
        field_dims: List of cardinalities per feature field
        embedding_dim: Embedding dimension per field
        n_experts: Number of expert networks
        expert_dim: Output dimension of each expert
        tower_dims: Hidden dims for task towers
        dropout: Dropout rate
    """
    
    def __init__(self, field_dims, embedding_dim, n_experts, expert_dim,
                 tower_dims, dropout=0.2):
        super().__init__()
        self.n_experts = n_experts
        
        # Shared embeddings
        self.embeddings = nn.ModuleList([
            nn.Embedding(d, embedding_dim, padding_idx=0) for d in field_dims
        ])
        
        input_dim = len(field_dims) * embedding_dim
        
        # Expert networks (shared across tasks)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, expert_dim), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(expert_dim, expert_dim), nn.ReLU()
            ) for _ in range(n_experts)
        ])
        
        # Task-specific gates
        self.gate_ctr = nn.Linear(input_dim, n_experts)
        self.gate_cvr = nn.Linear(input_dim, n_experts)
        
        # Task towers
        self.tower_ctr = make_tower(expert_dim, tower_dims, dropout)
        self.tower_cvr = make_tower(expert_dim, tower_dims, dropout)
        
        # Store gating weights for analysis
        self._last_ctr_gate = None
        self._last_cvr_gate = None
    
    def forward(self, feat_ids, feat_vals=None):
        """Forward pass with ESMM-style CTR x CVR multiplication."""
        x = embed_features(self.embeddings, feat_ids, feat_vals)
        
        # All experts process the same input
        expert_outs = torch.stack(
            [expert(x) for expert in self.experts], dim=1
        )  # (B, n_experts, expert_dim)
        
        # Task-specific gating
        g_ctr = torch.softmax(self.gate_ctr(x), dim=-1).unsqueeze(-1)  # (B, E, 1)
        g_cvr = torch.softmax(self.gate_cvr(x), dim=-1).unsqueeze(-1)
        
        # Store for analysis
        self._last_ctr_gate = g_ctr.squeeze(-1).detach()
        self._last_cvr_gate = g_cvr.squeeze(-1).detach()
        
        # Gated expert mixture
        ctr_input = (expert_outs * g_ctr).sum(dim=1)  # (B, expert_dim)
        cvr_input = (expert_outs * g_cvr).sum(dim=1)
        
        # Predictions
        ctr = torch.sigmoid(self.tower_ctr(ctr_input).squeeze(-1))
        cvr = torch.sigmoid(self.tower_cvr(cvr_input).squeeze(-1))
        ctcvr = ctr * cvr
        
        return ctr, cvr, ctcvr


# Create MMoE model
model_mmoe = MMoE(
    field_dims, EMBEDDING_DIM, N_EXPERTS, EXPERT_DIM, TOWER_DIMS, DROPOUT
).to(device)
n_params_mmoe = sum(p.numel() for p in model_mmoe.parameters())
print(f'MMoE parameters: {n_params_mmoe:,}')
print(f'  Experts: {N_EXPERTS} x {EXPERT_DIM}D')
print(f'  Towers: {TOWER_DIMS}')

## 5. PLE Implementation

### 5.1 Extraction Network

In [ ]:
class PLEExtractionLayer(nn.Module):
    """Single PLE extraction layer.
    
    Contains:
    - Shared experts (used by all tasks)
    - Task-specific experts (one set per task)
    - Task-specific gating networks
    
    Args:
        input_dim: Input dimension
        expert_dim: Expert output dimension
        n_shared: Number of shared experts
        n_task_specific: Number of task-specific experts per task
        n_tasks: Number of tasks (2 for CTR + CVR)
        dropout: Dropout rate
    """
    
    def __init__(self, input_dim, expert_dim, n_shared, n_task_specific,
                 n_tasks=2, dropout=0.2):
        super().__init__()
        self.n_tasks = n_tasks
        total_experts = n_shared + n_task_specific
        
        # Shared experts
        self.shared_experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, expert_dim), nn.ReLU(), nn.Dropout(dropout)
            ) for _ in range(n_shared)
        ])
        
        # Task-specific experts
        self.task_experts = nn.ModuleList([
            nn.ModuleList([
                nn.Sequential(
                    nn.Linear(input_dim, expert_dim), nn.ReLU(), nn.Dropout(dropout)
                ) for _ in range(n_task_specific)
            ]) for _ in range(n_tasks)
        ])
        
        # Task-specific gates
        self.gates = nn.ModuleList([
            nn.Linear(input_dim, total_experts) for _ in range(n_tasks)
        ])
    
    def forward(self, task_inputs):
        """Process task inputs through extraction layer.
        
        Args:
            task_inputs: list of tensors, one per task
        
        Returns:
            list of gated expert outputs, one per task
        """
        # Shared experts process first task's input
        shared_outs = [expert(task_inputs[0]) for expert in self.shared_experts]
        
        outputs = []
        for t in range(self.n_tasks):
            # Task-specific experts
            task_outs = [expert(task_inputs[t]) for expert in self.task_experts[t]]
            # Combine shared + task-specific
            all_outs = torch.stack(shared_outs + task_outs, dim=1)  # (B, total_E, D)
            # Gate
            g = torch.softmax(self.gates[t](task_inputs[t]), dim=-1).unsqueeze(-1)
            outputs.append((all_outs * g).sum(dim=1))  # (B, D)
        
        return outputs

In [ ]:
class PLE(nn.Module):
    """Progressive Layered Extraction for CVR prediction.
    
    Uses ESMM-style CTCVR = CTR x CVR with progressive extraction layers.
    
    Architecture:
        Embeddings -> [ExtractionLayer1 -> ... -> ExtractionLayerL]
        -> Tower_CTR -> ctr_pred
        -> Tower_CVR -> cvr_pred
        -> ctcvr = ctr * cvr
    
    Args:
        field_dims: Feature field cardinalities
        embedding_dim: Embedding size per field
        n_layers: Number of extraction layers
        expert_dim: Expert output dimension
        n_shared: Shared experts per layer
        n_task_specific: Task-specific experts per task per layer
        tower_dims: Tower hidden dimensions
        dropout: Dropout rate
    """
    
    def __init__(self, field_dims, embedding_dim, n_layers, expert_dim,
                 n_shared, n_task_specific, tower_dims, dropout=0.2):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(d, embedding_dim, padding_idx=0) for d in field_dims
        ])
        
        input_dim = len(field_dims) * embedding_dim
        
        # Progressive extraction layers
        self.extraction_layers = nn.ModuleList()
        prev_dim = input_dim
        for _ in range(n_layers):
            self.extraction_layers.append(
                PLEExtractionLayer(
                    prev_dim, expert_dim, n_shared, n_task_specific, 2, dropout
                )
            )
            prev_dim = expert_dim
        
        # Task towers
        self.tower_ctr = make_tower(expert_dim, tower_dims, dropout)
        self.tower_cvr = make_tower(expert_dim, tower_dims, dropout)
    
    def forward(self, feat_ids, feat_vals=None):
        """Forward pass with progressive extraction."""
        x = embed_features(self.embeddings, feat_ids, feat_vals)
        
        # Both tasks start from the same input
        task_inputs = [x, x]
        
        # Progressive extraction
        for layer in self.extraction_layers:
            task_inputs = layer(task_inputs)
        
        # Task predictions
        ctr = torch.sigmoid(self.tower_ctr(task_inputs[0]).squeeze(-1))
        cvr = torch.sigmoid(self.tower_cvr(task_inputs[1]).squeeze(-1))
        ctcvr = ctr * cvr
        
        return ctr, cvr, ctcvr


# Create PLE model
model_ple = PLE(
    field_dims, EMBEDDING_DIM, n_layers=2, expert_dim=EXPERT_DIM,
    n_shared=2, n_task_specific=2, tower_dims=TOWER_DIMS, dropout=DROPOUT
).to(device)
n_params_ple = sum(p.numel() for p in model_ple.parameters())
print(f'PLE parameters: {n_params_ple:,}')
print(f'  Extraction layers: 2')
print(f'  Shared experts: 2 per layer')
print(f'  Task-specific experts: 2 per task per layer')

## 6. Training

> **Pro Tip:** MMoE and PLE use the same training procedure as ESMM. The only difference
> is the model architecture -- the loss function (CTR + CTCVR) remains identical. This
> makes these models drop-in replacements for the ESMM towers.

In [ ]:
def evaluate_model(model, data_loader, device=device):
    """Evaluate multi-task model."""
    model.eval()
    all_ctr, all_cvr, all_ctcvr = [], [], []
    all_clicks, all_convs = [], []
    
    with torch.no_grad():
        for batch in data_loader:
            ids, vals, clicks, convs = [b.to(device) for b in batch]
            ctr, cvr, ctcvr = model(ids, vals)
            all_ctr.append(ctr.cpu().numpy())
            all_cvr.append(cvr.cpu().numpy())
            all_ctcvr.append(ctcvr.cpu().numpy())
            all_clicks.append(clicks.cpu().numpy())
            all_convs.append(convs.cpu().numpy())
    
    ctr_p = np.concatenate(all_ctr)
    cvr_p = np.concatenate(all_cvr)
    ctcvr_p = np.concatenate(all_ctcvr)
    clicks = np.concatenate(all_clicks)
    convs = np.concatenate(all_convs)
    
    metrics = {
        'ctr_auc': roc_auc_score(clicks, ctr_p),
        'ctcvr_auc': roc_auc_score(convs, ctcvr_p) if convs.sum() > 0 else 0.0,
    }
    clicked = clicks == 1
    if clicked.sum() > 10 and convs[clicked].sum() > 0:
        metrics['cvr_auc'] = roc_auc_score(convs[clicked], cvr_p[clicked])
    else:
        metrics['cvr_auc'] = 0.0
    
    return metrics


def train_model(model, train_loader, val_loader, model_name,
                n_epochs=5, lr=1e-3, weight_decay=1e-5, patience=3):
    """Train a multi-task model (MMoE or PLE)."""
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    history = {
        'train_loss': [], 'ctr_auc': [], 'cvr_auc': [], 'ctcvr_auc': []
    }
    best_ctcvr = 0
    patience_counter = 0
    
    print(f'\nTraining {model_name} ({sum(p.numel() for p in model.parameters()):,} params)...')
    
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        n_batches = 0
        start = time.time()
        
        for batch in train_loader:
            ids, vals, clicks, convs = [b.to(device) for b in batch]
            optimizer.zero_grad()
            
            ctr, cvr, ctcvr = model(ids, vals)
            
            ctr_loss = nn.functional.binary_cross_entropy(ctr, clicks)
            ctcvr_loss = nn.functional.binary_cross_entropy(ctcvr, convs)
            loss = ctr_loss + ctcvr_loss
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
        
        metrics = evaluate_model(model, val_loader, device)
        elapsed = time.time() - start
        
        history['train_loss'].append(total_loss / n_batches)
        for k in ['ctr_auc', 'cvr_auc', 'ctcvr_auc']:
            history[k].append(metrics[k])
        
        if metrics['ctcvr_auc'] > best_ctcvr:
            best_ctcvr = metrics['ctcvr_auc']
            patience_counter = 0
        else:
            patience_counter += 1
        
        print(f'  Epoch {epoch+1}/{n_epochs} ({elapsed:.1f}s) | '
              f'Loss: {total_loss/n_batches:.4f} | '
              f'CTR: {metrics["ctr_auc"]:.4f} | '
              f'CVR: {metrics["cvr_auc"]:.4f} | '
              f'CTCVR: {metrics["ctcvr_auc"]:.4f}')
        
        if patience_counter >= patience:
            print(f'  Early stopping at epoch {epoch+1}')
            break
    
    return history

In [ ]:
# Train MMoE
print("=" * 80)
print("TRAINING MMoE")
print("=" * 80)

mmoe_history = train_model(
    model_mmoe, train_loader, val_loader, model_name='MMoE',
    n_epochs=NUM_EPOCHS, lr=LEARNING_RATE, weight_decay=1e-5, patience=3
)

In [ ]:
# Train PLE
print("\n" + "=" * 80)
print("TRAINING PLE")
print("=" * 80)

ple_history = train_model(
    model_ple, train_loader, val_loader, model_name='PLE',
    n_epochs=NUM_EPOCHS, lr=LEARNING_RATE, weight_decay=1e-5, patience=3
)

## 7. Comparison with ESMM

In [ ]:
# Load ESMM results from Notebook 02
esmm_results_loaded = None
if (PROCESSED_DIR / 'esmm_results.json').exists():
    with open(PROCESSED_DIR / 'esmm_results.json', 'r') as f:
        esmm_saved = json.load(f)
    esmm_results_loaded = esmm_saved.get('esmm', {})
    naive_results_loaded = esmm_saved.get('naive_cvr', {})
    print('Loaded ESMM results from Notebook 02')
else:
    print('ESMM results not found. Using placeholder values.')
    esmm_results_loaded = {'ctr_auc': 0.6696, 'cvr_auc': 0.6518, 'ctcvr_auc': 0.6177}
    naive_results_loaded = {'cvr_auc': 0.5868, 'ctcvr_auc': 0.5385}

# Current model results
mmoe_results = evaluate_model(model_mmoe, val_loader, device)
ple_results = evaluate_model(model_ple, val_loader, device)

# Comparison table
print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)
print(f'{"Model":<15} {"CTR AUC":>10} {"CVR AUC":>10} {"CTCVR AUC":>10}')
print('-' * 50)

for name, res in [('Naive CVR', naive_results_loaded),
                   ('ESMM', esmm_results_loaded),
                   ('MMoE', mmoe_results),
                   ('PLE', ple_results)]:
    ctr_s = f'{res.get("ctr_auc", 0):.4f}' if res.get('ctr_auc') else '  -'
    cvr_s = f'{res.get("cvr_auc", 0):.4f}'
    ctcvr_s = f'{res.get("ctcvr_auc", 0):.4f}'
    print(f'{name:<15} {ctr_s:>10} {cvr_s:>10} {ctcvr_s:>10}')

In [ ]:
# Training curve comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
ax = axes[0]
ax.plot(mmoe_history['train_loss'], 'b-o', label='MMoE', markersize=4)
ax.plot(ple_history['train_loss'], 'r-s', label='PLE', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# CTR AUC
ax = axes[1]
ax.plot(mmoe_history['ctr_auc'], 'b-o', label='MMoE', markersize=4)
ax.plot(ple_history['ctr_auc'], 'r-s', label='PLE', markersize=4)
if esmm_results_loaded:
    ax.axhline(y=esmm_results_loaded.get('ctr_auc', 0), color='green',
               linestyle='--', label=f'ESMM: {esmm_results_loaded.get("ctr_auc", 0):.4f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('AUC')
ax.set_title('CTR AUC', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# CTCVR AUC
ax = axes[2]
ax.plot(mmoe_history['ctcvr_auc'], 'b-o', label='MMoE', markersize=4)
ax.plot(ple_history['ctcvr_auc'], 'r-s', label='PLE', markersize=4)
if esmm_results_loaded:
    ax.axhline(y=esmm_results_loaded.get('ctcvr_auc', 0), color='green',
               linestyle='--', label=f'ESMM: {esmm_results_loaded.get("ctcvr_auc", 0):.4f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('AUC')
ax.set_title('CTCVR AUC', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('MMoE vs PLE Training on Ali-CCP', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 8. Expert Utilization Analysis

> **Concept:** Understanding which experts are used by which task reveals how the model
> partitions knowledge. In a well-trained MMoE, different tasks should weight experts
> differently, indicating task-specific feature extraction.

In [ ]:
# Collect gating weights from MMoE
model_mmoe.eval()
all_ctr_gates = []
all_cvr_gates = []

with torch.no_grad():
    for batch in val_loader:
        ids, vals = batch[0].to(device), batch[1].to(device)
        _ = model_mmoe(ids, vals)
        all_ctr_gates.append(model_mmoe._last_ctr_gate.cpu().numpy())
        all_cvr_gates.append(model_mmoe._last_cvr_gate.cpu().numpy())
        if len(all_ctr_gates) >= 20:  # Enough samples
            break

ctr_gates = np.concatenate(all_ctr_gates)
cvr_gates = np.concatenate(all_cvr_gates)

# Expert utilization visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Average gate weights
ax = axes[0]
x = np.arange(N_EXPERTS)
width = 0.35
ax.bar(x - width/2, ctr_gates.mean(axis=0), width, label='CTR Task', color='#4e79a7')
ax.bar(x + width/2, cvr_gates.mean(axis=0), width, label='CVR Task', color='#e15759')
ax.set_xlabel('Expert Index')
ax.set_ylabel('Average Gate Weight')
ax.set_title('Expert Utilization by Task', fontsize=13)
ax.set_xticks(x)
ax.legend()

# Gate weight distributions
ax = axes[1]
for i in range(N_EXPERTS):
    ax.hist(ctr_gates[:, i], bins=50, alpha=0.5, label=f'Expert {i}')
ax.set_xlabel('Gate Weight')
ax.set_ylabel('Frequency')
ax.set_title('CTR Gate Weight Distribution', fontsize=13)
ax.legend(fontsize=9)

# Gate correlation heatmap
ax = axes[2]
gate_matrix = np.zeros((N_EXPERTS, 2))
gate_matrix[:, 0] = ctr_gates.mean(axis=0)
gate_matrix[:, 1] = cvr_gates.mean(axis=0)
sns.heatmap(gate_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=['CTR', 'CVR'],
            yticklabels=[f'Expert {i}' for i in range(N_EXPERTS)],
            ax=ax)
ax.set_title('Expert-Task Affinity', fontsize=13)

plt.suptitle('MMoE Expert Gating Analysis', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Task similarity via gate correlation
gate_corr = np.corrcoef(ctr_gates.mean(axis=0), cvr_gates.mean(axis=0))[0, 1]
print(f'\nGate correlation between CTR and CVR tasks: {gate_corr:.4f}')
print('(High correlation = tasks use similar expert mixtures)')
print('(Low correlation = tasks specialize on different experts)')

## 9. Ablation Studies

In [ ]:
# Ablation 1: Number of experts in MMoE
print("=" * 70)
print("ABLATION 1: Number of Experts (MMoE)")
print("=" * 70)

expert_results = {}

for n_exp in [2, 4, 8]:
    print(f'\n--- n_experts = {n_exp} ---')
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    
    m = MMoE(field_dims, EMBEDDING_DIM, n_exp, EXPERT_DIM, TOWER_DIMS, DROPOUT).to(device)
    h = train_model(m, train_loader, val_loader, f'MMoE-{n_exp}exp',
                    n_epochs=NUM_EPOCHS, patience=5)
    r = evaluate_model(m, val_loader, device)
    r['params'] = sum(p.numel() for p in m.parameters())
    expert_results[n_exp] = r

In [ ]:
# Ablation 2: Number of PLE extraction layers
print("\n" + "=" * 70)
print("ABLATION 2: Number of Extraction Layers (PLE)")
print("=" * 70)

layer_results = {}

for n_layers in [1, 2, 3]:
    print(f'\n--- n_layers = {n_layers} ---')
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    
    m = PLE(field_dims, EMBEDDING_DIM, n_layers, EXPERT_DIM,
            2, 2, TOWER_DIMS, DROPOUT).to(device)
    h = train_model(m, train_loader, val_loader, f'PLE-{n_layers}L',
                    n_epochs=NUM_EPOCHS, patience=5)
    r = evaluate_model(m, val_loader, device)
    r['params'] = sum(p.numel() for p in m.parameters())
    layer_results[n_layers] = r

In [ ]:
# Visualize ablation results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Experts ablation
ax = axes[0]
n_exps = list(expert_results.keys())
ctcvr_aucs = [expert_results[n]['ctcvr_auc'] for n in n_exps]
cvr_aucs = [expert_results[n]['cvr_auc'] for n in n_exps]

ax.plot(n_exps, ctcvr_aucs, 'r-o', label='CTCVR AUC', linewidth=2, markersize=8)
ax.plot(n_exps, cvr_aucs, 'b--s', label='CVR AUC', linewidth=2, markersize=8)
ax.set_xlabel('Number of Experts')
ax.set_ylabel('AUC')
ax.set_title('MMoE: Effect of Number of Experts', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# Layers ablation
ax = axes[1]
n_layers_list = list(layer_results.keys())
ctcvr_aucs_l = [layer_results[n]['ctcvr_auc'] for n in n_layers_list]
cvr_aucs_l = [layer_results[n]['cvr_auc'] for n in n_layers_list]

ax.plot(n_layers_list, ctcvr_aucs_l, 'r-o', label='CTCVR AUC', linewidth=2, markersize=8)
ax.plot(n_layers_list, cvr_aucs_l, 'b--s', label='CVR AUC', linewidth=2, markersize=8)
ax.set_xlabel('Number of Extraction Layers')
ax.set_ylabel('AUC')
ax.set_title('PLE: Effect of Extraction Depth', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Save models and results
torch.save(model_mmoe.state_dict(), str(PROCESSED_DIR / 'mmoe_model.pt'))
torch.save(model_ple.state_dict(), str(PROCESSED_DIR / 'ple_model.pt'))

def convert_for_json(obj):
    if isinstance(obj, (np.floating, np.float64, np.float32)): return float(obj)
    if isinstance(obj, (np.integer, np.int64, np.int32)): return int(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

results_to_save = {
    'mmoe': {
        'ctr_auc': convert_for_json(mmoe_results['ctr_auc']),
        'cvr_auc': convert_for_json(mmoe_results['cvr_auc']),
        'ctcvr_auc': convert_for_json(mmoe_results['ctcvr_auc']),
        'params': n_params_mmoe,
    },
    'ple': {
        'ctr_auc': convert_for_json(ple_results['ctr_auc']),
        'cvr_auc': convert_for_json(ple_results['cvr_auc']),
        'ctcvr_auc': convert_for_json(ple_results['ctcvr_auc']),
        'params': n_params_ple,
    },
}

with open(PROCESSED_DIR / 'advanced_cvr_results.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)

print('Models and results saved.')

---

## Exercises

### Exercise 1: Expert Dropout
Implement "expert dropout" where during training, each expert is randomly dropped with
probability 0.1. Does this improve generalization?

In [ ]:
# TODO: Exercise 1
# Modify MMoE forward to randomly zero out expert outputs during training
# Hint: Use torch.bernoulli to create a mask over expert outputs

pass

### Exercise 2: Task-Specific Learning Rates
Try using different learning rates for shared components vs task-specific components.
Use 1e-3 for shared embeddings/experts and 1e-4 for task towers.

In [ ]:
# TODO: Exercise 2
# Use parameter groups in the optimizer:
# optimizer = torch.optim.Adam([
#     {'params': shared_params, 'lr': 1e-3},
#     {'params': tower_params, 'lr': 1e-4},
# ])

pass

### Exercise 3: Gating Diversity Loss
Add a regularization term that encourages different tasks to use different gating patterns.
Implement $\mathcal{L}_{div} = -\|g_{ctr} - g_{cvr}\|_2^2$ and add it to the total loss.

In [ ]:
# TODO: Exercise 3
# Modify train loop to add diversity regularization
# L_div = -lambda * ||avg_ctr_gate - avg_cvr_gate||^2
# This encourages the tasks to specialize on different experts

pass

### Exercise 4: Hybrid Architecture
Create a hybrid model that combines MMoE-style gating with PLE-style task-specific experts.
Compare its performance against both MMoE and PLE.

In [ ]:
# TODO: Exercise 4
# Design a model with:
# - Shared experts (like MMoE)
# - Task-specific experts (like PLE)
# - Single extraction layer (no progressive stacking)

pass

---

## Summary & Key Takeaways

### What We Learned

1. **MMoE provides flexible task routing**: Unlike ESMM's shared-bottom approach, MMoE allows each task to dynamically select which expert knowledge to use. This is beneficial when CTR and CVR tasks need different feature representations.

2. **PLE reduces negative transfer**: By adding task-specific experts alongside shared experts, PLE ensures each task has dedicated capacity. The progressive extraction layers refine representations through multiple stages.

3. **Expert gating reveals task relationships**: The gating analysis shows how CTR and CVR tasks differ in their use of shared experts. High gate correlation suggests the tasks share common patterns; low correlation indicates specialization.

4. **More experts is not always better**: The ablation study shows that model performance can plateau or even decrease with too many experts, due to the difficulty of learning many expert-gate combinations with limited data.

5. **On Ali-CCP**: Reference results -- MMoE: CTR AUC ~0.669, CVR AUC ~0.618, CTCVR AUC ~0.607; PLE: CTR AUC ~0.668, CVR AUC ~0.628, CTCVR AUC ~0.606.

### Next Steps

In the final notebook, we consolidate all model results for a comprehensive head-to-head comparison with statistical analysis, business impact estimation, and deployment recommendations.